# Battery SOH Real Dataset Evaluation
This notebook loads the uploaded battery cycle dataset and evaluates a leakage-safe battery split with LSTM and XGBoost.

In [4]:
import json, math
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
import xgboost as xgb

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

repo = Path(r'c:/Users/supri/Downloads/AutoCircle-AI-Productionnn/AutoCircle-AI')
results_dir = repo / 'results'
models_dir = results_dir / 'models'
predictions_dir = results_dir / 'predictions'
for output_dir in (results_dir, models_dir, predictions_dir):
    output_dir.mkdir(exist_ok=True, parents=True)

def metrics(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return {
        'MAE': float(mean_absolute_error(y_true, y_pred)),
        'RMSE': float(math.sqrt(mean_squared_error(y_true, y_pred))),
        'R2': float(r2_score(y_true, y_pred)),
    }

dataset_path = repo / 'data' / 'battery_cycle_level_dataset_CLEAN_FINAL.csv'
df = pd.read_csv(dataset_path)
print('dataset', dataset_path.name)
print('rows', len(df), 'batteries', df['battery_id'].nunique())
print(df.isna().sum().to_dict())
print('dups', int(df.duplicated().sum()))

feature_cols = ['cycle', 'voltage', 'temperature', 'capacity', 'rul']
target_col = 'soh'

battery_ids = sorted(df['battery_id'].unique())
train_ids, temp_ids = train_test_split(battery_ids, test_size=0.2, random_state=SEED)
val_ids, test_ids = train_test_split(temp_ids, test_size=0.5, random_state=SEED)
train_df = df[df['battery_id'].isin(train_ids)].copy()
val_df = df[df['battery_id'].isin(val_ids)].copy()
test_df = df[df['battery_id'].isin(test_ids)].copy()

print('train/val/test rows', len(train_df), len(val_df), len(test_df))

# LSTM sequence prep
seq_len = 10

def prepare_sequence_data(frame):
    xs, ys = [], []
    for battery_id, group in frame.groupby('battery_id'):
        group = group.sort_values('cycle').reset_index(drop=True)
        if len(group) < seq_len:
            continue
        X = group[feature_cols].to_numpy(dtype=np.float32)
        y = group[target_col].to_numpy(dtype=np.float32)
        for idx in range(len(group) - seq_len + 1):
            xs.append(X[idx:idx + seq_len])
            ys.append(y[idx + seq_len - 1])
    return np.asarray(xs, dtype=np.float32), np.asarray(ys, dtype=np.float32)

class SeqDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).view(-1, 1)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

class LSTMModel(nn.Module):
    def __init__(self, input_dim=5):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_dim, hidden_size=64, num_layers=2, batch_first=True, dropout=0.2)
        self.fc = nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 1))
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

Xtrain, ytrain = prepare_sequence_data(train_df)
Xval, yval = prepare_sequence_data(val_df)
Xtest, ytest = prepare_sequence_data(test_df)
print('lstm sequences', Xtrain.shape, Xval.shape, Xtest.shape)

train_ds = SeqDataset(Xtrain, ytrain)
val_ds = SeqDataset(Xval, yval)
test_ds = SeqDataset(Xtest, ytest)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

model = LSTMModel()
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
best_state = None
best_val = float('inf')
for epoch in range(30):
    model.train()
    for xb, yb in train_loader:
        optimizer.zero_grad()
        pred = model(xb)
        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()
    model.eval()
    total = 0.0
    n = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            pred = model(xb)
            total += criterion(pred, yb).item() * len(xb)
            n += len(xb)
    val_loss = total / max(1, n)
    if val_loss < best_val:
        best_val = val_loss
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
model.load_state_dict(best_state)
model.eval()
lstm_preds = []
with torch.no_grad():
    for xb, _ in test_loader:
        lstm_preds.append(model(xb).squeeze(-1).cpu().numpy())
lstm_preds = np.concatenate(lstm_preds)
lstm_metrics = metrics(ytest, lstm_preds)
print('LSTM metrics', lstm_metrics)

# XGBoost
X_train = train_df[feature_cols].to_numpy(dtype=np.float32)
y_train = train_df[target_col].to_numpy(dtype=np.float32)
X_val = val_df[feature_cols].to_numpy(dtype=np.float32)
y_val = val_df[target_col].to_numpy(dtype=np.float32)
X_test = test_df[feature_cols].to_numpy(dtype=np.float32)
y_test = test_df[target_col].to_numpy(dtype=np.float32)

xgb_model = xgb.XGBRegressor(
    objective='reg:squarederror',
    n_estimators=500,
    max_depth=5,
    learning_rate=0.03,
    subsample=0.85,
    colsample_bytree=0.85,
    random_state=SEED,
)
xgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
xgb_pred = xgb_model.predict(X_test)
xgb_metrics = metrics(y_test, xgb_pred)
print('XGB metrics', xgb_metrics)

torch.save(model.state_dict(), models_dir / 'lstm_real_dataset.pt')
xgb_model.save_model(str(models_dir / 'xgboost_real_dataset.json'))
pd.DataFrame({
    'actual_soh': ytest,
    'predicted_soh': lstm_preds,
    'absolute_error': np.abs(ytest - lstm_preds),
}).to_csv(predictions_dir / 'lstm_test_predictions.csv', index=False)
pd.DataFrame({
    'actual_soh': y_test,
    'predicted_soh': xgb_pred,
    'absolute_error': np.abs(y_test - xgb_pred),
}).to_csv(predictions_dir / 'xgboost_test_predictions.csv', index=False)

(results_dir / 'battery_real_metrics.json').write_text(
    json.dumps({
        'dataset_file': dataset_path.name,
        'dataset_rows': int(len(df)),
        'unique_batteries': int(df['battery_id'].nunique()),
        'missing_values': {str(key): int(value) for key, value in df.isna().sum().items()},
        'duplicate_rows': int(df.duplicated().sum()),
        'features': feature_cols,
        'target': target_col,
        'seed': SEED,
        'split_method': 'battery-level train/validation/test split',
        'train_records': int(len(train_df)),
        'val_records': int(len(val_df)),
        'test_records': int(len(test_df)),
        'train_batteries': train_ids,
        'val_batteries': val_ids,
        'test_batteries': test_ids,
        'lstm_test_sequences': int(len(ytest)),
        'lstm': lstm_metrics,
        'xgboost': xgb_metrics,
        'claim_under_1_5_pct': bool(min(lstm_metrics['MAE'], xgb_metrics['MAE']) * 100 < 1.5),
    }, indent=2),
    encoding='utf-8',
)
print('saved models, predictions, and metrics')


dataset battery_cycle_level_dataset_CLEAN_FINAL.csv
rows 1415 batteries 34
{'battery_id': 0, 'cycle': 0, 'voltage': 0, 'temperature': 0, 'capacity': 0, 'soh': 0, 'rul': 0}
dups 0
train/val/test rows 1138 106 171
lstm sequences (968, 10, 5) (80, 10, 5) (150, 10, 5)
LSTM metrics {'MAE': 0.059310443699359894, 'RMSE': 0.0851678767053976, 'R2': 0.4236375093460083}
XGB metrics {'MAE': 0.08031409233808517, 'RMSE': 0.13642707092287235, 'R2': -0.30989551544189453}
saved models, predictions, and metrics
